# 03 · Race laps, weather and race control from TracingInsights (free)

Since October 2026, F1's timing APIs need an F1 TV login, so FastF1 can't download race lap data without a subscription (notebook 02). [TracingInsights](https://github.com/TracingInsights) published the same FastF1-derived data on GitHub, one repo per season, under the Apache-2.0 licence.

For each race this saves three files to Google Drive: **laps** (every lap of every driver, with tyre compound and age, stint, track status for safety car/VSC, pit in/out, accuracy flags and weather per lap), **weather** and **race control messages**.

Only those files are downloaded; the repos' telemetry, radio and cache folders are skipped. No rate limits apply, so one run covers all seasons. The repos are cloned to Colab's temporary disk, and only the parquet files go to Drive.

In [ ]:
# @title Setup: mount Drive, get the latest code, install FastF1
REPO_URL = "https://github.com/YOUR_USERNAME/f1-race-sim.git"  # @param {type:"string"}
DATA_DIR = "/content/drive/MyDrive/f1sim_data"  # @param {type:"string"}

import os, sys
from google.colab import drive

drive.mount("/content/drive")

REPO_DIR = "/content/" + REPO_URL.rstrip("/").split("/")[-1].removesuffix(".git")
if os.path.isdir(REPO_DIR):
    !git -C {REPO_DIR} pull --ff-only
else:
    !git clone {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
!pip install -q "fastf1>=3.9,<4"

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
os.environ["F1SIM_DATA_DIR"] = DATA_DIR

%load_ext autoreload
%autoreload 2
print("Code:", REPO_DIR, "| Data:", DATA_DIR)

In [ ]:
# @title Download (one run is enough)
START_YEAR = 2018  # @param {type:"integer"}
END_YEAR = 2025  # @param {type:"integer"}

from f1sim.paths import get_paths
from f1sim.ingest import quiet_fastf1_logs
from f1sim.ingest_tracing import ingest_tracing_seasons

quiet_fastf1_logs(True)
paths = get_paths()
summary = ingest_tracing_seasons(range(START_YEAR, END_YEAR + 1), paths)

In [ ]:
# @title Coverage per season
from f1sim.ingest_tracing import load_race_ti

laps = load_race_ti("laps", paths)
weather = load_race_ti("weather", paths)

per_race = laps.groupby(["Year", "Round", "Event"]).agg(
    Laps=("lap", "size"), Drivers=("drv", "nunique"),
    NoCompound=("compound", lambda s: int(s.isna().sum())),
).reset_index()
rain = weather.groupby(["Year", "Round"])["wR"].any().rename("AnyRain")
per_race = per_race.join(rain, on=["Year", "Round"])
per_race["AnyRain"] = per_race["AnyRain"].astype("boolean")
per_race["WeatherMissing"] = per_race["AnyRain"].isna().astype(int)

per_race.groupby("Year").agg(
    Races=("Round", "size"), Laps=("Laps", "sum"),
    MinDrivers=("Drivers", "min"), LapsNoCompound=("NoCompound", "sum"),
    RacesWithRain=("AnyRain", "sum"),
    WeatherMissing=("WeatherMissing", "sum"),
)

In [ ]:
# @title Tyre compounds per season
laps.groupby("Year")["compound"].value_counts(dropna=False).unstack(fill_value=0)

In [ ]:
# @title Races where rain was recorded at any point
per_race[per_race["AnyRain"] == True][["Year", "Round", "Event", "Laps"]]

In [ ]:
# @title Drive space used
!du -sh "{paths.race_ti_dir}"